In [0]:
catalog = "bank_risk"
schema = "bank_risk"
volume_path = f"/Volumes/{catalog}/{schema}/raw_data"

In [0]:
churn_raw = (spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{volume_path}/Churn-for-Bank-Customers.csv"))


In [0]:
fraud_raw = (spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{volume_path}/AIML Dataset.csv"))

fraud_raw.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.bronze_fraud")

In [0]:
from pyspark.sql import functions as F

catalog = "bank_risk"
schema = "bank_risk"

churn = churn_raw

churn_silver = (churn
    .drop("RowNumber", "Surname")   # not predictive; Surname is PII you don't want in a feature table
    .withColumn("BalanceSalaryRatio", F.col("Balance") / (F.col("EstimatedSalary") + F.lit(1)))
    .withColumn("ProductsPerTenure", F.col("NumOfProducts") / (F.col("Tenure") + F.lit(1)))
    .dropDuplicates(["CustomerId"])
)

churn_silver.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.silver_churn_features")

In [0]:
from pyspark.sql import Window

fraud = spark.table(f"{catalog}.{schema}.bronze_fraud")

client_window = Window.partitionBy("nameOrig")

fraud_silver = (fraud
    .withColumn("balanceDelta", F.col("oldbalanceOrg") - F.col("newbalanceOrig"))
    .withColumn("clientAvgAmount", F.avg("amount").over(client_window))
    .withColumn("clientStdAmount", F.stddev("amount").over(client_window))
    .withColumn(
        "amountZScore",
        (F.col("amount") - F.col("clientAvgAmount")) / (F.col("clientStdAmount") + F.lit(1))
    )
    .withColumn("isLargeTransfer", (F.col("type") == "TRANSFER") & (F.col("amount") > 200000))
)

fraud_silver.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.silver_fraud_features")

In [0]:
catalog = "bank_risk"
schema = "bank_risk"

churn_gold = spark.table(f"{catalog}.{schema}.silver_churn_features").select(
    "CustomerId", "CreditScore", "Geography", "Gender", "Age", "Tenure",
    "Balance", "NumOfProducts", "HasCrCard", "IsActiveMember",
    "EstimatedSalary", "BalanceSalaryRatio", "ProductsPerTenure",
    F.col("Exited").alias("label")
)
churn_gold.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.gold_churn")

In [0]:
fraud_gold = (spark.table(f"{catalog}.{schema}.silver_fraud_features")
    .groupBy("nameOrig")
    .agg(
        F.count("*").alias("txnCount"),
        F.avg("amount").alias("avgAmount"),
        F.stddev("amount").alias("stdAmount"),
        F.max("amountZScore").alias("maxZScore"),
        F.sum(F.col("isLargeTransfer").cast("int")).alias("largeTransferCount"),
        F.max("isFraud").alias("label")   # client is flagged if ANY of their transactions was fraud
    )
    .fillna(0)
)
fraud_gold.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.gold_fraud")

In [0]:
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import BinaryClassificationEvaluator

catalog = "bank_risk"
schema = "bank_risk"

churn_gold = spark.table(f"{catalog}.{schema}.gold_churn")

geo_indexer = StringIndexer(inputCol="Geography", outputCol="GeographyIdx")
gender_indexer = StringIndexer(inputCol="Gender", outputCol="GenderIdx")

feature_cols = ["CreditScore", "GeographyIdx", "GenderIdx", "Age", "Tenure",
                 "Balance", "NumOfProducts", "HasCrCard", "IsActiveMember",
                 "EstimatedSalary", "BalanceSalaryRatio", "ProductsPerTenure"]

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
rf = RandomForestClassifier(labelCol="label", featuresCol="features", numTrees=100)
pipeline = Pipeline(stages=[geo_indexer, gender_indexer, assembler, rf])

train, test = churn_gold.randomSplit([0.8, 0.2], seed=42)
churn_model = pipeline.fit(train)
preds = churn_model.transform(test)

evaluator = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderPR")
print("Churn PR-AUC:", evaluator.evaluate(preds))

Churn PR-AUC: 0.6730252211809584


In [0]:
fraud_gold = spark.table(f"{catalog}.{schema}.gold_fraud")

fraud_ratio = fraud_gold.filter("label = 1").count() / fraud_gold.count()
fraud_gold = fraud_gold.withColumn(
    "classWeight",
    F.when(F.col("label") == 1, 1 - fraud_ratio).otherwise(fraud_ratio)
)

feature_cols_fraud = ["txnCount", "avgAmount", "stdAmount", "maxZScore", "largeTransferCount"]
assembler_fraud = VectorAssembler(inputCols=feature_cols_fraud, outputCol="features")
rf_fraud = RandomForestClassifier(labelCol="label", featuresCol="features",
                                   weightCol="classWeight", numTrees=100)
pipeline_fraud = Pipeline(stages=[assembler_fraud, rf_fraud])

train_f, test_f = fraud_gold.randomSplit([0.8, 0.2], seed=42)
fraud_model = pipeline_fraud.fit(train_f)
preds_f = fraud_model.transform(test_f)

evaluator_f = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderPR")
print("Fraud PR-AUC:", evaluator_f.evaluate(preds_f))

Fraud PR-AUC: 0.20139647317721665


In [0]:
from pyspark.ml.functions import vector_to_array

churn_scores = churn_model.transform(churn_gold).select(
    "CustomerId",
    vector_to_array("probability")[1].alias("risk_score"),
    "prediction"
)
churn_scores.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{schema}.risk_scores_churn")

fraud_scores = fraud_model.transform(fraud_gold).select(
    "nameOrig",
    vector_to_array("probability")[1].alias("risk_score"),
    "prediction"
)
fraud_scores.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{schema}.risk_scores_fraud")

In [0]:
from sklearn.ensemble import IsolationForest

fraud_pd = spark.table(f"{catalog}.{schema}.gold_fraud").toPandas()
iso = IsolationForest(contamination=0.01, random_state=42)
fraud_pd["anomalyScore"] = iso.fit_predict(fraud_pd[feature_cols_fraud])

spark.createDataFrame(fraud_pd).write.mode("overwrite") \
    .saveAsTable(f"{catalog}.{schema}.gold_fraud_with_anomaly")